In [ ]:
import torch
import numpy as np
import pandas as pd
import yaml
import multiprocessing as mp
from torch_geometric.data import Data
from functools import partial
from easydict import EasyDict
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem.rdForceFieldHelpers import MMFFOptimizeMolecule

# from ..chem import set_rdmol_positions, get_best_rmsd
import pickle

import sys
# from utils.data_filter import filter_data
import argparse

In [ ]:
def print_covmat_results(results, print_fn=print):
    # Remove index 32 from inputs
    mask = results.MatchingP < 1
    print(np.sum(results.MatchingP >= 1))

    CoverageR = results.CoverageR[mask]
    CoverageP = results.CoverageP[mask]
    MatchingR = results.MatchingR[mask]
    MatchingP = results.MatchingP[mask]

    # Core COV/ MAT results
    df = pd.DataFrame({
        'COV-R_mean':   np.nanmean(CoverageR, 0),
        'COV-R_median': np.nanmedian(CoverageR, 0),
        'COV-R_std':    np.nanstd(CoverageR, 0),
        'COV-P_mean':   np.nanmean(CoverageP, 0),
        'COV-P_median': np.nanmedian(CoverageP, 0),
        'COV-P_std':    np.nanstd(CoverageP, 0),
    }, index=results.thresholds)
    print_fn('\n' + str(df))

    # MAT-R and MAT-P
    mat_r_mean   = np.nanmean(MatchingR)
    mat_r_med    = np.nanmedian(MatchingR)
    mat_r_std    = np.nanstd(MatchingR)
    mat_p_mean   = np.nanmean(MatchingP)
    mat_p_med    = np.nanmedian(MatchingP)
    mat_p_std    = np.nanstd(MatchingP)
    print_fn(f'MAT-R_mean: {mat_r_mean:.4f} | MAT-R_median: {mat_r_med:.4f} | MAT-R_std {mat_r_std:.4f}')
    print_fn(f'MAT-P_mean: {mat_p_mean:.4f} | MAT-P_median: {mat_p_med:.4f} | MAT-P_std {mat_p_std:.4f}')

    # NaN statistics (per molecule, from results.NanCounts) — also drop index 32 if present
    nan_counts = np.array(list(results.NanCounts.values()), dtype=float)
    if len(nan_counts) > 32:
        nan_counts = np.delete(nan_counts, 32)
    nan_mean   = np.mean(nan_counts)
    nan_med    = np.median(nan_counts)
    nan_std    = np.std(nan_counts)
    print_fn(f'\nNaN counts per molecule — mean: {nan_mean:.2f}, median: {nan_med:.2f}, std: {nan_std:.2f}')

    # Number of skipped SMILES
    num_skipped = len(results.SkippedSmiles)
    print_fn(f'Number of skipped SMILES (insufficient valid conformers): {num_skipped}')

    # NaN counts directly in COV / MAT arrays
    covr_nans = np.isnan(CoverageR).sum()
    covp_nans = np.isnan(CoverageP).sum()
    matr_nans = np.isnan(MatchingR).sum()
    matp_nans = np.isnan(MatchingP).sum()
    total_nans = covr_nans + covp_nans + matr_nans + matp_nans
    print_fn(
        f'\nNaN entries in metrics — '
        f'COV-R: {covr_nans}, COV-P: {covp_nans}, '
        f'MAT-R: {matr_nans}, MAT-P: {matp_nans}, '
        f'Total: {total_nans}'
    )

    return df

In [ ]:
    file = '../model_outputs_official/qm9_conformer_official_fixed/qm9_noH_1000_kabsch_traj_interpolator_pretrain_final_539_25_simple_conf_actual/epoch=399-step=69200-lambda-0.200-gen-eval.json'

In [ ]:
with open(file, 'rb') as f:
    run_data = pickle.load(f)

In [ ]:
run_data.keys()

In [ ]:
print_covmat_results(run_data)